# **2110433 - Computer Vision (2024/2)**
#**Lab 8 - Convolutional Neural Network [Homework]** <br>
In this lab, we will learn how to use Convolutional Neural Network to perform image classification in the provided real world dataset using PyTorch. This notebook includes both coding and written questions. Please hand in this notebook file with all outputs and your answer.

**Collaboration is encouraged in this course.** You must turn in your own write ups of all problems. If you collaborate with others, you must put the names and ids of the students you worked with in below block.

Collaboration List:
- ...
- ...


# Assignment 1 : Food Image Classification

Classify 50 food menus from Chula-Food-50 dataset
 

In this assignment you have to replace YOUR_STUDENT_ID_WITH21 variable with your student id (in integer). There will be 2 sets of data: train and test 

By using the knowledge from the lab and lecture, you have to design your own CNN food image classification model and tested on unknown label dataset!



Scoreboard URL : https://www.piclab.ai/classes/cv2024/lab8/scoreboard

In [1]:
%pip install -q tqdm torch numpy matplotlib torchvision requests lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 20.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
##### Add Thai font ######
import matplotlib
from matplotlib import font_manager

!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf -O thsarabunnew-webfont.ttf
font_manager.fontManager.addfont(path="thsarabunnew-webfont.ttf")
matplotlib.rc("font", family="TH Sarabun New")

In [3]:
import random
import glob
import os
import numpy as np
import cv2
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger


from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision import models as models
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

IS_ON_COULD = False
MAX_EPOCH = 1000

TRAIN_FOLDER_PATH = "/kaggle/input/cu-food-50/train"

### Split train and validation ###
TEST_SIZE = 0.1
SEED = 42
BATCH_SIZE = 64

# Classifier hyperparameters
NUM_CLASSES = 50
N_LAYERS = 7
DROPOUT_RATE = 0.3

# Training hyperparameters
LEARNING_RATE = 1e-3
ES_PATIENCE = 5
LRS_FACTOR = 0.5
LRS_PATIENCE = 3

# Transform
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

YOUR_STUDENT_ID_WITH21 = 6432023321

## Your model description goes here: ###
WRITE HERE

## Download and inspect Chula-Food-50 dataset

In [4]:
if IS_ON_COULD:
    !wget  -O chula-food-50.zip https://piclab.ai/classes/cv2021/Chula-food-50.zip
    !unzip -qo chula-food-50.zip

In [5]:
#### FILL Any Augmenetation HERE ####
transformTrain = transforms.Compose(
    [
        transforms.Resize(size=(288, 288), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize([0, 0, 0], [255, 255, 255]),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ]
)

transformVal = transforms.Compose(
    [
        transforms.Resize(size=(288, 288), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize([0, 0, 0], [255, 255, 255]),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ]
)

### Load Dataset ###
foodTrainDataset = ImageFolder(TRAIN_FOLDER_PATH, transform=transformTrain)

In [6]:
# generate indices: instead of the actual data we pass in integers instead
train_indices, test_indices, _, _ = train_test_split(
    range(len(foodTrainDataset)),
    foodTrainDataset.targets,
    stratify=foodTrainDataset.targets,
    test_size=TEST_SIZE,
    random_state=SEED,
)

# generate subset based on indices
train_split = Subset(foodTrainDataset, train_indices)
test_split = Subset(foodTrainDataset, test_indices)

## Define CNN network for food classification
Hint
1. You can freely uses any structure/pretrained model to do this homework but don't forgot to cited them in this notebook.

   A very big collection of pretrained model can be found here : https://github.com/rwightman/pytorch-image-models

2. Don't forget to change mean and std in the pre-processing to match with your pretrained model.

In [7]:
#### Design you network here ####
class foodNet(LightningModule):
    def __init__(
        self,
    ):
        super(foodNet, self).__init__()
        self.save_hyperparameters()
        # Load model
        self.model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)

        # Freeze all layers except the final classification layer
        for param in self.model.parameters():
            param.requires_grad = False

        # Change the final classification layer
        embed_size = self.model.classifier[-1].in_features
        self.model.classifier = nn.Sequential()
        for layer_index in range(N_LAYERS):
            self.model.classifier.add_module(
                f"layer_{layer_index}",
                nn.Linear(embed_size, embed_size),
            )
            self.model.classifier.add_module(
                f"leaky_relu_{layer_index}",
                nn.LeakyReLU()
            )
            self.model.classifier.add_module(
                f"dropout_{layer_index}",
                nn.Dropout(DROPOUT_RATE)
            )
            # Add batch normalization every 3 layers
            if layer_index % 3 == 0:
                self.model.classifier.add_module(
                    f"batch_norm_{layer_index}",
                    nn.BatchNorm1d(embed_size)
                )
        self.model.classifier.add_module("output", nn.Linear(embed_size, NUM_CLASSES))

        self.softmax = nn.Softmax(dim=1)

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input):
        ### Conntections goes here ###
        embedding = self.model(input)
        return self.softmax(embedding)

    def training_step(self, batch, batch_idx):
        ### Training Logic goes here ###
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)

        # Compute accuracy
        _, predicted = torch.max(y_hat, 1)
        correct = (predicted == y).sum().item()
        self.log("train_acc", correct / y.size(0), on_step=True, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        ### Validation Logic goes here ###
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("val_loss", loss, on_step=True, on_epoch=True, prog_bar=True)

        # Compute accuracy
        _, predicted = torch.max(y_hat, 1)
        correct = (predicted == y).sum().item()
        self.log("val_acc", correct / y.size(0), on_step=True, on_epoch=True, prog_bar=True)

        return loss

    def configure_optimizers(self):
        ### Optimizer goes here ###
        optimizer = torch.optim.Adam(self.parameters(), lr=LEARNING_RATE)
        lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=LRS_FACTOR, patience=LRS_PATIENCE, verbose=True
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": lr_scheduler,
            "monitor": "val_loss",
        }

## Construct the model, optimizer and loss function

In [8]:
#### FILL HERE ####
net = foodNet()

foodTrainDatasetLoader = DataLoader(train_split, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
foodValDatasetLoader = DataLoader(test_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth
100%|██████████| 35.2M/35.2M [00:00<00:00, 153MB/s]


## Train the model

In [9]:
# Define Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    dirpath="./model/",
    filename="foodnet-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
    mode="min",
)

early_stop_callback = EarlyStopping(monitor="val_loss", patience=ES_PATIENCE, verbose=True, mode="min")

lr_monitor = LearningRateMonitor(logging_interval="step")

# Define Logger
logger = CSVLogger("logs", name="foodnet")

# Define Trainer
trainer = Trainer(max_epochs=MAX_EPOCH, callbacks=[checkpoint_callback, early_stop_callback, lr_monitor], logger=logger)

# Start Training
trainer.fit(net, foodTrainDatasetLoader, foodValDatasetLoader)

### Load the best model ###
best_model_path = checkpoint_callback.best_model_path
best_model = foodNet.load_from_checkpoint(best_model_path)

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## Classify on validation set and send result to server!

In [10]:
from PIL import Image
import json
import requests


class ImageFolderWithPaths(Dataset):
    def __init__(self, root_dir, transform=None):
        self.imageFileNames = sorted(glob.glob(root_dir + "/*.jpg"))
        self.transform = transform

    def __getitem__(self, idx):
        imageData = Image.open(self.imageFileNames[idx])
        imageFileName = os.path.basename(self.imageFileNames[idx])
        if self.transform is not None:
            imageData = self.transform(imageData)
        return imageFileName, imageData.unsqueeze(0)

    def __len__(self):
        return len(self.imageFileNames)


def generatePredictedResults(valDataset, net):
    net.eval()
    predictedResults = {}
    with torch.no_grad():
        for imageFileName, imageData in tqdm(valDataset):
            if torch.cuda.is_available():
                imageData = imageData.cuda()
            outputs = net(imageData)
            _, predicted = torch.max(outputs, 1)
            # print(imageFileName, predicted.item())
            predictedResults[imageFileName] = foodTrainDataset.classes[predicted.item()]
    return predictedResults


def sendResult(predictedResults, studentID):
    sendDict = {"studentID": studentID, "results": predictedResults}
    response = requests.post(
        "https://www.piclab.ai/classes/cv2024/lab8/scoreboard/submit",
        headers={"Content-Type": "application/json"},
        json=sendDict,
    )
    return response.text

In [11]:
foodTestDataset = ImageFolderWithPaths("/kaggle/input/cu-food-50/test", transform=transformVal)
predictedResults = generatePredictedResults(foodTestDataset, best_model)
print(sendResult(predictedResults, studentID=YOUR_STUDENT_ID_WITH21))

  0%|          | 0/5000 [00:00<?, ?it/s]

{"accuracy":"27.66","status":"SUCCESS"}

